# 01_dataset — Peek inside a LeRobotDataset

Load LeRobot v0.5.1 / `LeRobotDataset` (v3.0 format), check shapes, and visualize an image.

Prerequisite: `config.env` has been `source`d (env vars `DATA_REPO`, `HF_HOME` are visible).
If the compute node is offline, pre-download (`env/predownload_hf.sh`) must be done.

In [ ]:
import os

# Use values from config.env (don't hard-code). Fail fast if unset.
REPO_ID = os.environ.get("DATA_REPO")
assert REPO_ID and not REPO_ID.startswith("<TODO"), (
    "DATA_REPO is unset. Run `source config.env` first."
)
# In an offline environment, uncomment to use only the pre-downloaded cache.
# os.environ["HF_HUB_OFFLINE"] = "1"
print("DATA_REPO =", REPO_ID)
print("HF_HOME   =", os.environ.get("HF_HOME"))

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(REPO_ID)

# Metadata: fps / number of episodes / number of frames
print("fps          =", dataset.meta.fps)
print("num_episodes =", dataset.num_episodes)
print("num_frames   =", dataset.num_frames)

In [ ]:
# Pull one frame and see what keys/tensors exist
sample = dataset[0]
for key, value in sample.items():
    shape = tuple(value.shape) if hasattr(value, "shape") else value
    print(f"{key:40s} {shape}")

In [ ]:
# Explicitly check the action and state shapes
print("action shape =", tuple(sample["action"].shape))
if "observation.state" in sample:
    print("state  shape =", tuple(sample["observation.state"].shape))

In [ ]:
# Visualize one camera image
import matplotlib.pyplot as plt

image_keys = [k for k in sample if k.startswith("observation.images")]
print("image keys:", image_keys)

if image_keys:
    img = sample[image_keys[0]]
    # LeRobot image tensors are (C, H, W). Convert to (H, W, C) for matplotlib.
    img_hwc = img.permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(4, 4))
    plt.imshow(img_hwc)
    plt.title(image_keys[0])
    plt.axis("off")
    plt.show()

## Take a time window with delta_timestamps

Specifying `delta_timestamps` (seconds relative to the current frame t) adds a time
axis `T` to the same key. This is where you build intuition for how ACT / Diffusion
Policy handle "multiple steps of observations and actions".

In [ ]:
if image_keys:
    key = image_keys[0]
    delta_timestamps = {key: [-0.2, -0.1, 0.0]}  # 0.2s/0.1s before + current
    ds_t = LeRobotDataset(REPO_ID, delta_timestamps=delta_timestamps)
    s = ds_t[100]
    print(f"{key} shape with time window =", tuple(s[key].shape))  # expect: (T, C, H, W), T=3